# Fake News Detection with ModernBERT  
**Objective**: Classify news articles into human-written vs. LLM-generated fake/real content using a two-stage BERT model.  
**Workflow**:  
1. Data Acquisition & Preprocessing  
2. Binary Classification (Fake vs. Real)  
3. Fine-Grained Classification (Human vs. GPT)  
4. Two-Stage Inference Pipeline  


**Model Architecture: Two-Stage Classification**

A. Stage 1 (Binary Classification):
- Detects fake (0) vs real (1) news using all data

B. Stage 2 (Source Identification):
- For fake articles: Distinguish human_fake vs gpt_fake
- For real articles: Distinguish human_real vs gpt_real

Data Loading → Cleaning → Balancing → Splitting  
  ↓  
Stage 1 Training → Evaluation  
  ↓  
Stage 2 Training → Evaluation  
  ↓  
Validation  

# 1. Environment Setup

## 1.1 Install & Import Dependencies  
*Import libraries and mount Google Drive for data access.*

In [1]:
# --- Core Libraries ---
import json  # For JSON file handling
import re  # Regular expressions for text cleaning
import numpy as np  # Numerical operations
import pandas as pd  # Data manipulation
import matplotlib.pyplot as plt  # Visualization
import seaborn as sns  # Enhanced visualizations

# --- Machine Learning & NLP ---
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix, precision_recall_fscore_support
)

# --- PyTorch & Transformers ---
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification, AutoConfig,
    Trainer, TrainingArguments, EarlyStoppingCallback
)

# --- Google Colab Integration ---
from google.colab import drive
drive.mount('/content/drive')  # Mount Google Drive

## 1.2 Helper Functions  
*Define reusable functions for data loading, cleaning, and preprocessing.*

### FakeNewsNet Dataset  
**Description**:  
- Combines human-written (HF/HR) and LLM-generated (MF/MR) news from [FakeNewsNet](https://github.com/KaiDMML/FakeNewsNet).  
- **Sources**:  
  - **PolitiFact**: Fact-checked political news  
  - **GossipCop**: Entertainment news  
- **Labels**:  
  - `human_fake`, `human_real`, `gpt_fake`, `gpt_real`  

**Preprocessing**:  
- Excluded articles missing titles/descriptions  
- SMP prompting via ChatGPT generated synthetic samples  

In [3]:
# Function to read a JSON file
def load_json_file(file_path):
    """Load a JSON file"""
    with open(file_path, "r", encoding="utf-8") as file:
        return json.load(file)

In [4]:
# Function to process FakeNewsNet data
def process_fakenewsnet_data(file_paths):
    """Read multiple JSON files and organize them into a DataFrame"""
    data_list = []
    label_map = {"HF": "human_fake", "HR": "human_real", "MF": "gpt_fake", "MR": "gpt_real"}

    for category, file_path in file_paths.items():
        data = load_json_file(file_path)
        source = "gossipcop" if "gossipcop" in category else "politifact"
        label = label_map.get(category.split("_")[-1], "unknown")

        for news_id, news_content in data.items():
            if 'text' in news_content:
                data_list.append({
                    "id": news_id,
                    "text": news_content["text"],
                    "title": news_content.get("title", ""),
                    "source": source,
                    "label": label,
                })

    return pd.DataFrame(data_list)


In [10]:
def clean_text(text: str) -> str:
    """Normalizes text by:
    1. Lowercasing
    2. Removing URLs
    3. Trimming extra whitespace

    Args:
        text (str): Raw input text

    Returns:
        str: Cleaned text
    """
    text = text.lower()
    text = re.sub(r"http\S+|www\S+", "", text)  # Remove URLs
    text = re.sub(r'\s+', ' ', text).strip()  # Collapse whitespace
    return text

 # 2. Data Exploration

## 2.1 Load Dataset  
*Load and preview the FakeNewsNet dataset.*

In [8]:
# Define dataset paths
fake_news_paths = {category: f"/content/drive/MyDrive/ColabNotebooks/W266/FinalProject/FakeNewsNet/{category}.json"
                    for category in ["gossipcop_HF", "gossipcop_HR", "politifact_HF", "politifact_HR",
                                     "gossipcop_MF", "gossipcop_MR", "politifact_MF", "politifact_MR"]}

# Load and process the data
FakeNewsNet_df = process_fakenewsnet_data(fake_news_paths)

# Display the first 5 rows
FakeNewsNet_df.head()

,id,text,title,source,label
0,0,✕ Close Meghan Markle and Prince Harry have an...,As it happened: Prince Harry and Meghan Markle...,gossipcop,human_fake
1,1,Kim Kardashian and Kanye West are pulling out ...,Kim & Kanye Install At-Home Panic Room After P...,gossipcop,human_fake
2,2,Prince Harry and Meghan currently live at Kens...,£1.4million spent renovating Prince Harry and ...,gossipcop,human_fake
3,3,They can't get enough of the Biebs on this sho...,Photos from Dancing With the Stars: Special Gu...,gossipcop,human_fake
4,4,Ben Affleck is keeping life with his three kid...,Jennifer Garner ‘Doesn’t Want’ Her Kids Around...,gossipcop,human_fake


## 2.2 Exploratory Analysis  
*Analyze class distribution and text statistics.*

In [9]:
# check if there is any missing value
FakeNewsNet_df.isnull().sum()

,0
id,0
text,0
title,0
source,0
label,0


In [ ]:
FakeNewsNet_df['label'].value_counts()

# 3. Data Preprocessing

## 3.1 Text Cleaning  
*Normalize text by removing URLs, lowercasing, and trimming whitespace.*

In [12]:
# Apply the text cleaning function
FakeNewsNet_df["cleaned_text"] = FakeNewsNet_df["text"].apply(clean_text)
# Display the first 5 rows
FakeNewsNet_df.head()

,id,text,title,source,label,cleaned_text
0,0,✕ Close Meghan Markle and Prince Harry have an...,As it happened: Prince Harry and Meghan Markle...,gossipcop,human_fake,✕ close meghan markle and prince harry have an...
1,1,Kim Kardashian and Kanye West are pulling out ...,Kim & Kanye Install At-Home Panic Room After P...,gossipcop,human_fake,kim kardashian and kanye west are pulling out ...
2,2,Prince Harry and Meghan currently live at Kens...,£1.4million spent renovating Prince Harry and ...,gossipcop,human_fake,prince harry and meghan currently live at kens...
3,3,They can't get enough of the Biebs on this sho...,Photos from Dancing With the Stars: Special Gu...,gossipcop,human_fake,they can't get enough of the biebs on this sho...
4,4,Ben Affleck is keeping life with his three kid...,Jennifer Garner ‘Doesn’t Want’ Her Kids Around...,gossipcop,human_fake,ben affleck is keeping life with his three kid...


## 3.2 Class Balancing  
*Undersample majority classes to address imbalance.*

In [13]:
# Filter the relevant columns
FakeNewsNet_df = FakeNewsNet_df[["cleaned_text", "label"]]

In [15]:
# balance the data
human_real_df = FakeNewsNet_df[FakeNewsNet_df['label'] == 'human_real'].sample(n=4181, random_state=42)
gpt_real_df = FakeNewsNet_df[FakeNewsNet_df['label'] == 'gpt_real'].sample(n=4181, random_state=42)
human_fake_df = FakeNewsNet_df[FakeNewsNet_df['label'] == 'human_fake']
gpt_fake_df = FakeNewsNet_df[FakeNewsNet_df['label'] == 'gpt_fake']

In [ ]:
# Before balancing
print("Original Class Distribution:")
print(FakeNewsNet_df['label'].value_counts())

# After balancing
print("\nBalanced Class Distribution:")
print(pd.concat([human_real_df, gpt_real_df, human_fake_df, gpt_fake_df])['label'].value_counts())

## 3.3 Train-Validation-Test Split  
*Split data into 80-10-10 stratified sets.*

In [16]:
# Split each label dataset into train (80%), validation (10%), and test (10%)
def split_dataset(df, label_name):
    train, temp = train_test_split(df, test_size=0.2, random_state=42, stratify=df['label'])
    val, test = train_test_split(temp, test_size=0.5, random_state=42, stratify=temp['label'])
    return train, val, test

# Apply the split function to each label category
human_real_train, human_real_val, human_real_test = split_dataset(human_real_df, "human_real")
human_fake_train, human_fake_val, human_fake_test = split_dataset(human_fake_df, "human_fake")
gpt_real_train, gpt_real_val, gpt_real_test = split_dataset(gpt_real_df, "gpt_real")
gpt_fake_train, gpt_fake_val, gpt_fake_test = split_dataset(gpt_fake_df, "gpt_fake")

# Stack the train, validation, and test sets
train_df = pd.concat([human_real_train, human_fake_train, gpt_fake_train, gpt_real_train]).sample(frac=1, random_state=42)
val_df = pd.concat([human_real_val, human_fake_val, gpt_fake_val, gpt_real_val]).sample(frac=1, random_state=42)
test_df = pd.concat([human_real_test, human_fake_test, gpt_fake_test, gpt_real_test]).sample(frac=1, random_state=42)

# Print label distributions
print("Training set label distribution:")
print(train_df['label'].value_counts())
print("\nValidation set label distribution:")
print(val_df['label'].value_counts())
print("\nTest set label distribution:")
print(test_df['label'].value_counts())

# Print dataset sizes
print(f"Training: {len(train_df)}, Validation: {len(val_df)}, Test: {len(test_df)}")

Training set label distribution:
label
gpt_fake      3344
human_real    3344
gpt_real      3344
human_fake    3344
Name: count, dtype: int64

Validation set label distribution:
label
human_fake    418
gpt_real      418
human_real    418
gpt_fake      418
Name: count, dtype: int64

Test set label distribution:
label
gpt_fake      419
gpt_real      419
human_real    419
human_fake    419
Name: count, dtype: int64
Training: 13376, Validation: 1672, Test: 1676


# 4. Model Development

### ModernBERT Architecture  
ModernBERT improves on BERT with:  
1. **Long-Context Handling**: Processes sequences up to 520 tokens.  
2. **Efficient Tokenization**: Better subword tokenization for noisy text.  
3. **Domain Adaptation**: Pretrained on diverse web text, ideal for fake news detection.  

## 4.1 Initialize Model & Tokenizer  
*Load ModernBERT and configure for sequence classification.*

In [17]:
# Tokenizer and model function
def load_model_and_tokenizer(model_name, num_labels):
    """Load tokenizer and model for a given model name."""
    tokenizer = AutoTokenizer.from_pretrained(model_name)

    config = AutoConfig.from_pretrained(model_name)
    config.num_labels = num_labels

    model = AutoModelForSequenceClassification.from_pretrained(model_name, config=config)

    return tokenizer, model, config.model_type  # 返回 model_type

## 4.2 Dataset & Training Setup  
*Tokenize text and create PyTorch datasets.*

In [ ]:
# Tokenization function
def tokenize_and_pad(texts, tokenizer, model_type="bert", max_length=512):
    """Tokenize and pad the text data, ensuring compatibility with different model architectures."""
    return_token_type_ids = model_type in ["bert", "deberta"]

    encoding = tokenizer(
        texts,
        truncation=True,  # Ensure texts longer than max_length are truncated
        padding="max_length",  # Pad texts to max_length
        max_length=max_length,
        return_tensors="pt",
        return_token_type_ids=return_token_type_ids  # Only return token_type_ids for BERT/DeBERTa
    )
    return encoding

In [18]:
# PyTorch Dataset class
class FakeNewsDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels, use_fp16):
        self.encodings = encodings
        self.labels = labels
        self.use_fp16 = use_fp16

    def __getitem__(self, idx):
        item = {
            key: val[idx].clone().detach()  # Fix tensor copy
            for key, val in self.encodings.items()
        }
        if self.use_fp16:
            item["attention_mask"] = item["attention_mask"].to(dtype=torch.float16)


        # Handle NaN values in labels:
        label_value = self.labels[idx]
        if np.isnan(label_value):  # Check if label is NaN
            # Replace NaN with a default value (e.g., -1) or handle it as needed
            label_value = -1

        # Convert labels to int before clamping
        # Ensure labels are within the valid range for int64
        item["labels"] = torch.tensor(int(label_value), dtype=torch.long).clamp(min=-(2**63), max=(2**63 - 1))
        return item

    def __len__(self):
        return len(self.labels)


In [19]:
def to_numeric_labels(labels, label_map):
    """Converts string labels to numeric labels using the provided label_map.
    Handles missing labels by assigning -1.
    """
    numeric_labels = []
    for label in labels:
        if isinstance(label, str):  # Check if label is a string
            if label in label_map:
                numeric_labels.append(label_map[label])
            else:
                numeric_labels.append(-1)
                print(f"Warning: Unrecognized string label '{label}' encountered. Assigning -1.")
        elif isinstance(label, int) or isinstance(label, np.integer): # Check if label is numeric
            if label in label_map.values():
                numeric_labels.append(label)  # Use the numeric label directly
            else:
                numeric_labels.append(-1)
                print(f"Warning: Unrecognized numeric label '{label}' encountered. Assigning -1.")
        else:
            numeric_labels.append(-1)
            print(f"Warning: Label '{label}' is of an unexpected type. Assigning -1.")
    return numeric_labels

## 4.3 Stage 1: Fake vs. Real Classification  
*Train binary classifier to detect fake news.*

In [20]:
def train_and_evaluate(
    model_name,
    num_labels,
    train_df,
    val_df,
    test_df,
    use_fp16=False,
    # Add custom parameters for Stage 2
    label_map=None,
    target_names=None
):
    """
    Orchestrates model training and evaluation
    Args:
        model_name: HF model identifier
        num_labels: 2 for binary, 4 for multi-class
        train_df: Training data with 'cleaned_text' and 'label'
        val_df: Validation data
        test_df: Test data
        use_fp16: Use mixed precision training
        label_map: Custom label mapping for Stage 2
        target_names: Custom target names for Stage 2
    """

    # Load tokenizer and model, and get model type
    tokenizer, model, model_type = load_model_and_tokenizer(model_name, num_labels)

    # Tokenization (pass model_type)
    train_encodings = tokenize_and_pad(train_df['cleaned_text'].tolist(), tokenizer, model_type=model_type)
    val_encodings = tokenize_and_pad(val_df['cleaned_text'].tolist(), tokenizer, model_type=model_type)
    test_encodings = tokenize_and_pad(test_df['cleaned_text'].tolist(), tokenizer, model_type=model_type)

    # Define label_map and target_names dynamically
    if label_map is None:
        if num_labels == 2:
            # Default for Stage 1: All fake=0, real=1
            label_map = {
                "human_fake": 0,
                "gpt_fake": 0,
                "human_real": 1,
                "gpt_real": 1
            }
            target_names = ["fake", "real"]  # Stage 1 names
        else:
            # Default for multi-class (unused in Stage 2)
            label_map = {
                "human_real": 0,
                "human_fake": 1,
                "gpt_real": 2,
                "gpt_fake": 3
            }
            target_names = list(label_map.keys())
    else:
        # Use custom label_map and target_names for Stage 2
        target_names = target_names if target_names is not None else list(label_map.keys())

    # Convert labels using the to_numeric_labels function:
    train_labels = to_numeric_labels(train_df['label'].tolist(), label_map)
    val_labels = to_numeric_labels(val_df['label'].tolist(), label_map)
    test_labels = to_numeric_labels(test_df['label'].tolist(), label_map)

    # Create Dataset
    train_dataset = FakeNewsDataset(train_encodings, train_labels, use_fp16)
    val_dataset = FakeNewsDataset(val_encodings, val_labels, use_fp16)
    test_dataset = FakeNewsDataset(test_encodings, test_labels, use_fp16)

    # Training Arguments
    training_args = TrainingArguments(
        output_dir="./results",
        logging_dir="./logs",
        logging_steps=100,
        eval_strategy="epoch",
        save_strategy="epoch",
        learning_rate=2e-5,
        per_device_train_batch_size=8,
        per_device_eval_batch_size=16,
        warmup_steps=500,
        num_train_epochs=2,
        weight_decay=0.01,
        load_best_model_at_end=True,
        metric_for_best_model="f1",
        report_to="none",
        fp16=True  # Enable mixed-precision training
        label_smoothing_factor=0.1,
        gradient_accumulation_steps=2  # Gradient accumulation
    )

    # Trainer
    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=val_dataset,
        compute_metrics=compute_metrics
    )

    # Train
    trainer.train()

    # Save Model
    model_path = f"/content/drive/MyDrive/ColabNotebooks/W266/FinalProject/model_checkpoints/{model_name}_base_model"
    trainer.save_model(model_path)

    # Evaluate
    eval_results = trainer.evaluate(test_dataset)
    print(f"Evaluation Results for {model_name}: {eval_results}")

    # Predict
    predictions = trainer.predict(test_dataset)
    y_pred = np.argmax(predictions.predictions, axis=-1)

    # Classification Report
    report = classification_report(
    test_labels,
    y_pred,
    target_names=target_names,  # Use dynamic names
    zero_division=0
    )
    print(f"Classification Report for {model_name}:\n{report}")

    # Confusion Matrix
    conf_matrix = confusion_matrix(test_labels, y_pred)
    print(f"Confusion Matrix for {model_name}:\n{conf_matrix}")

    # Misclassification Rate
    misclassification_rate = 1 - accuracy_score(test_labels, y_pred)
    print(f"Misclassification Rate for {model_name}: {misclassification_rate:.4f}")

    return {
    "model": model,  # Return the trained model
    "eval_results": eval_results,  # Validation metrics
    "test_report": report,  # Classification report
    "conf_matrix": conf_matrix,  # Confusion matrix
    "misclassification_rate": misclassification_rate  # Misclassification rate
    }

In [21]:
# Compute metrics function
def compute_metrics(eval_pred) -> dict:
    """Computes classification metrics for Hugging Face Trainer"""
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)

    return {
        "accuracy": accuracy_score(labels, preds),
        "precision": precision_score(labels, preds, average="weighted"),
        "recall": recall_score(labels, preds, average="weighted"),
        "f1": f1_score(labels, preds, average="weighted")
    }

In [22]:
def check_label_distribution(df, column_name="label"):
    """Checks and prints the distribution of labels in a DataFrame column,
    including NaN values.
    """
    label_counts = df[column_name].value_counts(dropna=False)  # Include NaN in counts
    print(label_counts)

In [ ]:
# Visualization: EDA plots (class distribution, text length histograms)
plt.figure(figsize=(10,6))
sns.countplot(x='label', data=FakeNewsNet_df)
plt.title("Class Distribution")
plt.show()

In [58]:
# --- Stage 1: Convert labels to binary (fake=0, real=1) ---
train_df_stage1 = train_df.copy()
train_df_stage1["label"] = train_df["label"].apply(lambda x: 0 if "fake" in x else 1)
val_df_stage1 = val_df.copy()
val_df_stage1["label"] = val_df["label"].apply(lambda x: 0 if "fake" in x else 1)
test_df_stage1 = test_df.copy()
test_df_stage1["label"] = test_df["label"].apply(lambda x: 0 if "fake" in x else 1)

# Check label distributions for each dataset
print("Training set label distribution:")
check_label_distribution(train_df_stage1)

print("\nValidation set label distribution:")
check_label_distribution(val_df_stage1)

print("\nTest set label distribution:")
check_label_distribution(test_df_stage1)

Training set label distribution:
label
0    6688
1    6688
Name: count, dtype: int64

Validation set label distribution:
label
0    836
1    836
Name: count, dtype: int64

Test set label distribution:
label
0    838
1    838
Name: count, dtype: int64


In [59]:
# Sample a small percentage (e.g., 1%) for testing
sample_size = 0.01  # Adjust this percentage as needed

train_df_sample = train_df_stage1.sample(frac=sample_size, random_state=42)
val_df_sample = val_df_stage1.sample(frac=sample_size, random_state=42)
test_df_sample = test_df_stage1.sample(frac=sample_size, random_state=42)

# Train Stage 1 using the sample datasets
stage1_results = train_and_evaluate(
    model_name="answerdotai/ModernBERT-base",
    num_labels=2,
    train_df=train_df_sample,  # Use the Stage 1 sample DataFrame
    val_df=val_df_sample,  # Use the Stage 1 sample DataFrame
    test_df=test_df_sample,  # Use the Stage 1 sample DataFrame
    use_fp16=True
)

Some weights of ModernBertForSequenceClassification were not initialized from the model checkpoint at answerdotai/ModernBERT-base and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,No log,0.692409,0.470588,0.592437,0.470588,0.489164


Evaluation Results for answerdotai/ModernBERT-base: {'eval_loss': 0.676670491695404, 'eval_accuracy': 0.5882352941176471, 'eval_precision': 0.6260504201680672, 'eval_recall': 0.5882352941176471, 'eval_f1': 0.5882352941176471, 'eval_runtime': 0.3119, 'eval_samples_per_second': 54.504, 'eval_steps_per_second': 9.618, 'epoch': 1.8235294117647058}
Classification Report for answerdotai/ModernBERT-base:
              precision    recall  f1-score   support

        fake       0.50      0.71      0.59         7
        real       0.71      0.50      0.59        10

    accuracy                           0.59        17
   macro avg       0.61      0.61      0.59        17
weighted avg       0.63      0.59      0.59        17

Confusion Matrix for answerdotai/ModernBERT-base:
[[5 2]
 [5 5]]
Misclassification Rate for answerdotai/ModernBERT-base: 0.4118


In [60]:
# Train Stage 1 and capture outputs
stage1_results = train_and_evaluate(
    model_name="answerdotai/ModernBERT-base",
    num_labels=2,
    train_df=train_df_stage1,
    val_df=val_df_stage1,
    test_df=test_df_stage1,
    use_fp16=True
)

# Save Stage 1 model
stage1_model = stage1_results["model"]
stage1_model.save_pretrained("/content/drive/MyDrive/Colab Notebooks/W266/FinalProject/stage1_model")

# Save metrics
with open("/content/drive/MyDrive/Colab Notebooks/W266/FinalProject/stage1_metrics.json", "w") as f:
    json.dump({
        "eval_results": stage1_results["eval_results"],
        "test_report": stage1_results["test_report"],
        "conf_matrix": stage1_results["conf_matrix"].tolist()  # Convert numpy array to list
    }, f)

Some weights of ModernBertForSequenceClassification were not initialized from the model checkpoint at answerdotai/ModernBERT-base and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,0.951300,0.393387,0.875000,0.884440,0.875000,0.874228
2,0.690200,0.374011,0.894737,0.896646,0.894737,0.894610


Evaluation Results for answerdotai/ModernBERT-base: {'eval_loss': 0.38209158182144165, 'eval_accuracy': 0.886038186157518, 'eval_precision': 0.8880945938787985, 'eval_recall': 0.886038186157518, 'eval_f1': 0.8858870227483775, 'eval_runtime': 21.6434, 'eval_samples_per_second': 77.437, 'eval_steps_per_second': 9.703, 'epoch': 2.0}
Classification Report for answerdotai/ModernBERT-base:
              precision    recall  f1-score   support

        fake       0.92      0.85      0.88       838
        real       0.86      0.92      0.89       838

    accuracy                           0.89      1676
   macro avg       0.89      0.89      0.89      1676
weighted avg       0.89      0.89      0.89      1676

Confusion Matrix for answerdotai/ModernBERT-base:
[[712 126]
 [ 65 773]]
Misclassification Rate for answerdotai/ModernBERT-base: 0.1140


## 4.4 Stage 2: Human vs. GPT Classification  
*Train specialized classifiers for fake/real subsets.*

In [63]:
# --- Stage 2: Split into fake/real subsets ---
# Fake subset (human_fake vs gpt_fake)
train_fake = train_df[train_df["label"].isin(["human_fake", "gpt_fake"])].copy()
train_fake["label"] = train_fake["label"].apply(lambda x: 0 if x == "human_fake" else 1)

# Real subset (human_real vs gpt_real)
train_real = train_df[train_df["label"].isin(["human_real", "gpt_real"])].copy()
train_real["label"] = train_real["label"].apply(lambda x: 0 if x == "human_real" else 1)

# Define custom label_map and target_names for Stage 2
label_map_fake = {"human_fake": 0, "gpt_fake": 1}
target_names_fake = ["human_fake", "gpt_fake"]

label_map_real = {"human_real": 0, "gpt_real": 1}
target_names_real = ["human_real", "gpt_real"]

# Train Fake Subset Model
stage2_fake_results = train_and_evaluate(
    model_name="answerdotai/ModernBERT-base",
    num_labels=2,
    train_df=train_fake,
    val_df=val_df[val_df["label"].isin(["human_fake", "gpt_fake"])],
    test_df=test_df[test_df["label"].isin(["human_fake", "gpt_fake"])],
    use_fp16=True,
    label_map=label_map_fake,          # Pass custom label_map
    target_names=target_names_fake     # Pass custom target_names
)

# Save Fake Subset Model
stage2_fake_model = stage2_fake_results["model"]
stage2_fake_model.save_pretrained("/content/drive/MyDrive/Colab Notebooks/W266/FinalProject/stage2_fake_model")

# Train Real Subset Model
stage2_real_results = train_and_evaluate(
    model_name="answerdotai/ModernBERT-base",
    num_labels=2,
    train_df=train_real,
    val_df=val_df[val_df["label"].isin(["human_real", "gpt_real"])],
    test_df=test_df[test_df["label"].isin(["human_real", "gpt_real"])],
    use_fp16=True,
    label_map=label_map_real,          # Pass custom label_map
    target_names=target_names_real     # Pass custom target_names
)

# Save Real Subset Model
stage2_real_model = stage2_real_results["model"]
stage2_real_model.save_pretrained("/content/drive/MyDrive/Colab Notebooks/W266/FinalProject/stage2_real_model")

# Save all Stage 2 metrics
stage2_metrics = {
    "fake_subset": {
        "eval_results": stage2_fake_results["eval_results"],
        "test_report": stage2_fake_results["test_report"],
        "conf_matrix": stage2_fake_results["conf_matrix"].tolist()
    },
    "real_subset": {
        "eval_results": stage2_real_results["eval_results"],
        "test_report": stage2_real_results["test_report"],
        "conf_matrix": stage2_real_results["conf_matrix"].tolist()
    }
}

with open("/content/drive/MyDrive/Colab Notebooks/W266/FinalProject/stage2_metrics.json", "w") as f:
    json.dump(stage2_metrics, f)

Some weights of ModernBertForSequenceClassification were not initialized from the model checkpoint at answerdotai/ModernBERT-base and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,No log,0.209493,0.994019,0.994045,0.994019,0.994019
2,0.474200,0.210127,0.995215,0.995261,0.995215,0.995215


Evaluation Results for answerdotai/ModernBERT-base: {'eval_loss': 0.2144126147031784, 'eval_accuracy': 0.9928400954653938, 'eval_precision': 0.9928513246409998, 'eval_recall': 0.9928400954653938, 'eval_f1': 0.9928400546821599, 'eval_runtime': 10.6796, 'eval_samples_per_second': 78.468, 'eval_steps_per_second': 9.832, 'epoch': 2.0}
Classification Report for answerdotai/ModernBERT-base:
              precision    recall  f1-score   support

  human_fake       1.00      0.99      0.99       419
    gpt_fake       0.99      1.00      0.99       419

    accuracy                           0.99       838
   macro avg       0.99      0.99      0.99       838
weighted avg       0.99      0.99      0.99       838

Confusion Matrix for answerdotai/ModernBERT-base:
[[415   4]
 [  2 417]]
Misclassification Rate for answerdotai/ModernBERT-base: 0.0072


Some weights of ModernBertForSequenceClassification were not initialized from the model checkpoint at answerdotai/ModernBERT-base and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,No log,0.355340,0.911483,0.915286,0.911483,0.911280
2,0.794500,0.332813,0.924641,0.928766,0.924641,0.924459


Evaluation Results for answerdotai/ModernBERT-base: {'eval_loss': 0.32909736037254333, 'eval_accuracy': 0.9272076372315036, 'eval_precision': 0.931337704163791, 'eval_recall': 0.9272076372315036, 'eval_f1': 0.9270329720524778, 'eval_runtime': 10.7164, 'eval_samples_per_second': 78.198, 'eval_steps_per_second': 9.798, 'epoch': 2.0}
Classification Report for answerdotai/ModernBERT-base:
              precision    recall  f1-score   support

  human_real       0.89      0.98      0.93       419
    gpt_real       0.97      0.88      0.92       419

    accuracy                           0.93       838
   macro avg       0.93      0.93      0.93       838
weighted avg       0.93      0.93      0.93       838

Confusion Matrix for answerdotai/ModernBERT-base:
[[409  10]
 [ 51 368]]
Misclassification Rate for answerdotai/ModernBERT-base: 0.0728


In [3]:
# --- Inference after training both stages ---
def two_stage_inference(text, stage1_model, stage2_fake_model, stage2_real_model, tokenizer):
    # Stage 1: Fake vs Real
    inputs = tokenizer(text, return_tensors="pt", padding=True, truncation=True, max_length=512)
    stage1_output = stage1_model(**inputs)
    stage1_pred = torch.argmax(stage1_output.logits, dim=1).item()  # 0=fake, 1=real

    # Stage 2: AI vs Human
    if stage1_pred == 0:
        stage2_output = stage2_fake_model(**inputs)
        stage2_pred = "human_fake" if torch.argmax(stage2_output.logits) == 0 else "gpt_fake"
    else:
        stage2_output = stage2_real_model(**inputs)
        stage2_pred = "human_real" if torch.argmax(stage2_output.logits) == 0 else "gpt_real"

    return stage1_pred, stage2_pred

In [5]:
!ls -l /content/drive/MyDrive/Colab_Notebooks/W266/FinalProject/stage1_model

total 584416
-rw------- 1 root root      1263 Apr  5 21:01 config.json
-rw------- 1 root root 598439784 Apr  5 21:01 model.safetensors


In [6]:
!pip install -U transformers huggingface_hub

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 127.9 MB/s eta 0:00:00
  Attempting uninstall: transformers
    Found existing installation: transformers 4.50.3
    Uninstalling transformers-4.50.3:
      Successfully uninstalled transformers-4.50.3


In [6]:
from transformers import AutoModelForSequenceClassification, AutoTokenizer

# Load Stage 1 Model
stage1_model = AutoModelForSequenceClassification.from_pretrained(
    "/content/drive/MyDrive/Colab_Notebooks/W266/FinalProject/stage1_model",
    local_files_only=True,  # Force local loading
    ignore_mismatched_sizes=True  # Suppress model size mismatch warnings
)

# Load Stage 2 Models
stage2_fake_model = AutoModelForSequenceClassification.from_pretrained(
    "/content/drive/MyDrive/Colab_Notebooks/W266/FinalProject/stage2_fake_model",
    local_files_only=True,
    ignore_mismatched_sizes=True
)

stage2_real_model = AutoModelForSequenceClassification.from_pretrained(
    "/content/drive/MyDrive/Colab_Notebooks/W266/FinalProject/stage2_real_model",
    local_files_only=True,
    ignore_mismatched_sizes=True
)

# Load tokenizer (use the original model name)
tokenizer = AutoTokenizer.from_pretrained("answerdotai/ModernBERT-base")

# Example inference
text = "BioVolve Therapeutics, a Silicon Valley biotech firm, faces lawsuits after internal documents revealed its gene-editing"
stage1_pred, stage2_pred = two_stage_inference(
    text, stage1_model, stage2_fake_model, stage2_real_model, tokenizer
)
print(f"Stage 1 (Fake/Real): {'Real' if stage1_pred == 1 else 'Fake'}")
print(f"Stage 2 (Source): {stage2_pred}")

Stage 1 (Fake/Real): Real
Stage 2 (Source): gpt_real


# 5.0 Validation

## Steps

### Data Source:
https://www.kaggle.com/datasets/clmentbisaillon/fake-and-real-news-dataset?resource=download&select=Fake.csv

### Data Loading & Preprocessing:

- Combines fake and real news datasets

- Adds binary labels ('fake'/'real')

- Cleans text (lowercasing, URL removal, whitespace normalization)

### Model Loading:

- Loads your pre-trained Stage 1 and Stage 2 models

- Uses the ModernBERT tokenizer

### Two-Stage Prediction:

- Stage 1: Classifies as fake/real

- Stage 2: For fake articles, classifies as human/GPT-generated fake
For real articles, classifies as human/GPT-generated real

### Evaluation:

- Provides classification report (precision, recall, F1-score)

- Shows confusion matrix

- Compares both binary (fake/real) and detailed (human/GPT) predictions

In [7]:
import pandas as pd
import numpy as np
import re
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

# Load and preprocess data
def load_and_preprocess_data():
    # Load datasets
    fake_df = pd.read_csv("/content/drive/MyDrive/Colab_Notebooks/W266/FinalProject/ISOT/Fake.csv")
    true_df = pd.read_csv("/content/drive/MyDrive/Colab_Notebooks/W266/FinalProject/ISOT/True.csv")

    # Add labels
    fake_df['label'] = 'fake'
    true_df['label'] = 'real'

    # Combine datasets
    df = pd.concat([fake_df, true_df], axis=0)
    df = df.sample(frac=1, random_state=42).reset_index(drop=True)

    # Clean text
    def clean_text(text):
        text = str(text).lower()
        text = re.sub(r'http\S+|www\S+', '', text)
        text = re.sub(r'\s+', ' ', text).strip()
        return text

    df['clean_text'] = df['text'].apply(clean_text)
    return df[['clean_text', 'label']]

# Load your pre-trained models
def load_models():
    # Stage 1 model (fake vs real)
    stage1_model = AutoModelForSequenceClassification.from_pretrained(
        "/content/drive/MyDrive/Colab_Notebooks/W266/FinalProject/stage1_model",
        local_files_only=True
    )

    # Stage 2 models
    stage2_fake_model = AutoModelForSequenceClassification.from_pretrained(
        "/content/drive/MyDrive/Colab_Notebooks/W266/FinalProject/stage2_fake_model",
        local_files_only=True
    )

    stage2_real_model = AutoModelForSequenceClassification.from_pretrained(
        "/content/drive/MyDrive/Colab_Notebooks/W266/FinalProject/stage2_real_model",
        local_files_only=True
    )

    tokenizer = AutoTokenizer.from_pretrained("answerdotai/ModernBERT-base")

    return stage1_model, stage2_fake_model, stage2_real_model, tokenizer

# Two-stage prediction function
def predict(text, models, tokenizer):
    stage1_model, stage2_fake_model, stage2_real_model = models

    # Tokenize input
    inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=512)

    # Stage 1 prediction
    stage1_output = stage1_model(**inputs)
    stage1_pred = torch.argmax(stage1_output.logits).item()  # 0=fake, 1=real

    # Stage 2 prediction
    if stage1_pred == 0:
        stage2_output = stage2_fake_model(**inputs)
        final_label = "fake_" + ["human", "gpt"][torch.argmax(stage2_output.logits).item()]
    else:
        stage2_output = stage2_real_model(**inputs)
        final_label = "real_" + ["human", "gpt"][torch.argmax(stage2_output.logits).item()]

    return final_label

# Main execution
if __name__ == "__main__":
    # Load and preprocess data
    df = load_and_preprocess_data()

    # Split data
    train_df, test_df = train_test_split(df, test_size=0.2, random_state=42)

    # Load models
    stage1_model, stage2_fake_model, stage2_real_model, tokenizer = load_models()
    models = (stage1_model, stage2_fake_model, stage2_real_model)

    # Run predictions
    test_texts = test_df['clean_text'].tolist()
    actual_labels = test_df['label'].tolist()

    predicted_labels = []
    for text in test_texts:
        predicted_labels.append(predict(text, models, tokenizer))

    # Convert labels to comparable format
    actual_binary = ['fake' if l == 'fake' else 'real' for l in actual_labels]
    predicted_binary = ['fake' if 'fake' in p else 'real' for p in predicted_labels]

    # Evaluate performance
    print("Classification Report:")
    print(classification_report(actual_binary, predicted_binary))

    print("\nConfusion Matrix:")
    print(confusion_matrix(actual_binary, predicted_binary, labels=['fake', 'real']))

Classification Report:
              precision    recall  f1-score   support

        fake       0.62      0.40      0.49      4710
        real       0.52      0.73      0.61      4270

    accuracy                           0.56      8980
   macro avg       0.57      0.56      0.55      8980
weighted avg       0.57      0.56      0.55      8980


Confusion Matrix:
[[1896 2814]
 [1170 3100]]
